In [ ]:
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

MAX_WORKERS = 20
RETRY = 2
TIMEOUT = 5  # Tăng timeout lên một chút để tránh bỏ sót các server phản hồi chậm

def load_urls(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"[!] Không tìm thấy file: {file_path}")
        return []

def normalize_url(url):
    url = url.strip()
    if not url.startswith("http://") and not url.startswith("https://"):
        url = "http://" + url
    return url

def is_valid_url(url):
    try:
        parsed = urlparse(url)
        return parsed.scheme in ("http", "https") and parsed.netloc != ""
    except Exception:
        return False

def check_alive(url):
    """
    Kiểm tra URL bằng phương thức GET nhưng dùng stream=True
    để không phải tải toàn bộ file (đặc biệt hữu ích với file ảnh, html nặng).
    """
    for _ in range(RETRY):
        try:
            res = requests.get(url, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True, stream=True) # có thể thêm verify=False để chấp nhận rủi ro lấy những trang bị cảnh báo 
            status = res.status_code
            res.close()  # Đóng kết nối ngay sau khi đọc được status code để tiết kiệm tài nguyên

            # Nếu mã trạng thái < 400 (200 OK, 301/302 Redirect...), coi như sống
            if status < 400:
                return "ALIVE"
            else:
                return "NOTFOUND"
                
        except requests.exceptions.RequestException:
            # Bỏ qua lỗi kết nối/timeout để thử lại lần tiếp theo (nếu còn RETRY)
            continue
            
    return "NOTFOUND"

def process_single(url):
    url = normalize_url(url)

    if not is_valid_url(url):
        return ("NOTFOUND", url)

    status = check_alive(url)
    return (status, url)

def process_urls(urls):
    alive = []
    notfound = []
    seen = set()

    print(f"[*] Bắt đầu kiểm tra {len(urls)} URLs với {MAX_WORKERS} luồng...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []

        for url in urls:
            url = normalize_url(url)
            # Lọc trùng lặp URL
            if url in seen:
                continue
            seen.add(url)
            futures.append(executor.submit(process_single, url))

        for future in as_completed(futures):
            status, url = future.result()

            if status == "ALIVE":
                print(f"[+] ALIVE: {url}")
                alive.append(url)
            else:
                print(f"[-] NOTFOUND: {url}")
                notfound.append(url)

    return alive, notfound

def save_urls(urls, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for url in urls:
            f.write(url + "\n")
    print(f"[*] Đã lưu {len(urls)} link vào {file_path}")

def main():
    # Thay đổi tên file cho khớp với file bạn xuất ra từ bước cào dữ liệu
    input_file = "full_urls.txt" 
    urls = load_urls(input_file)
    
    if not urls:
        print("[!] Không có URL nào để xử lý. Vui lòng kiểm tra lại.")
        return

    print(f"[*] Tổng số URL gốc: {len(urls)}")

    alive, notfound = process_urls(urls)

    print("\n--- TỔNG KẾT ---")
    print(f"Truy cập được (Alive): {len(alive)}")
    print(f"Không truy cập được (Not Found): {len(notfound)}")

    save_urls(alive, "urls_alive.txt")
    save_urls(notfound, "urls_notfound.txt")

if __name__ == "__main__":
    main()